In [2]:
import json
import os
import time
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm
from pyAutoSummarizer.base import psr
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from openai import OpenAI
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

In [3]:
from datasets import load_dataset
dataset = load_dataset("Awesome075/multi_news_parquet")

In [6]:
# Access the 'train' split of the dataset
train_data = dataset['train']

# Extract the 'document' column
documents = train_data['document']

# Extract the 'summary' column
summaries = train_data['summary']

print('First 5 documents:')
for i, doc in enumerate(documents[:5]):
    print(f"Document {i+1}:\n{doc[:200]}...\n") # Displaying first 200 characters for brevity

print('First 5 summaries:')
for i, summary in enumerate(summaries[:5]):
    print(f"Summary {i+1}:\n{summary[:]}...\n") # Displaying first 200 characters for brevity

First 5 documents:
Document 1:
National Archives 
 
 Yes, it’s that time again, folks. It’s the first Friday of the month, when for one ever-so-brief moment the interests of Wall Street, Washington and Main Street are all aligned o...

Document 2:
LOS ANGELES (AP) — In her first interview since the NBA banned her estranged husband, Shelly Sterling says she will fight to keep her share of the Los Angeles Clippers and plans one day to divorce Don...

Document 3:
GAITHERSBURG, Md. (AP) — A small, private jet has crashed into a house in Maryland's Montgomery County on Monday, killing at least three people on board, authorities said. 
 
 Preliminary information ...

Document 4:
Tucker Carlson Exposes His Own Sexism on Twitter (Updated) 
 
 Tucker Carlson has done some good work in the past… His site, The Daily Caller, is a frequent stop of mine and many other Conservatives. ...

Document 5:
A man accused of removing another man's testicle during a meeting in a Port Macquarie motel room has 

In [4]:
NUM_DOCS = 100
#MAX_WORKERS = 4
def summarize_document_with_qwen(documents):
    try:
        completion = client.chat.completions.create(
            model="qwen2.5-7b-instruct-mlx", # Replace with the exact model name shown in LM Studio
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes news articles from different sources concisely."},
                {
                    "role": "user",
                    "content":(
                    "/no_think\n\n"
                    f"Summarize the following document into a comprehensive summary: {documents}"
        )
        }

       ],
            max_tokens=200,
            temperature=0.3,
            extra_body={"enable_thinking": False}
        )
        choice = completion.choices[0]      # <-- this line was missing
        content = choice.message.content 
        #print("FULL MESSAGE:")
        #print(choice.message)


        if content is None:
            print(f"WARNING: empty content. finish_reason={choice.finish_reason}")
            return f"Error: empty response (finish_reason={choice.finish_reason})"
        return content   # <-- this was missing — the actual success return

        
    except Exception as e:
        print(f"Error summarizing document: {e}")
        return "Error: Could not generate summary."



In [1]:
# -----------------------------------------------------------------------
 #2. Generate summaries for the first NUM_DOCS documents
# -----------------------------------------------------------------------
print(f"Generating summaries for the first {NUM_DOCS} documents...")
 
generated_summaries = []
for i in tqdm(range(NUM_DOCS), desc="Summarizing documents"):
    doc = documents[i]
    summary = summarize_document_with_qwen(doc)
    generated_summaries.append(summary)
 

NameError: name 'NUM_DOCS' is not defined

In [ ]:
 #2. Generate summaries for the first NUM_DOCS documents
# -----------------------------------------------------------------------
print(f"Generating summaries for the first {NUM_DOCS} documents...")
 
generated_summaries = []
for i in tqdm(range(NUM_DOCS), desc="Summarizing documents"):
    doc = documents[i]
    summary = summarize_document_with_qwen(doc)
    generated_summaries.append(summary)
 

In [24]:
# 3. Reference (human) summaries — same slice, aligned by index
# -----------------------------------------------------------------------
summary_col = train_data['summary']
if hasattr(summary_col, 'tolist'):
    reference_summaries = summary_col[:num_documents_to_summarize].tolist()
else:
    reference_summaries = list(summary_col[:num_documents_to_summarize])
 
assert len(generated_summaries) == len(reference_summaries), \
    "Generated and reference summary counts don't match — check indexing."
 
# -----------------------------------------------------------------------
# 4. Evaluation
#    pyAutoSummarizer's metric methods (rouge_N, rouge_L, bleu, meteor)
#    take (generated, reference) directly, so the summarization object
#    just needs to be instantiated once — the init text isn't used by
#    these particular metric calls.
# -----------------------------------------------------------------------
s = psr.summarization("placeholder", stop_words=['en'], lowercase=True,
                       rmv_accents=True, rmv_special_chars=True, rmv_numbers=True)
 
rows = []
for i, (gen, ref) in enumerate(tqdm(zip(generated_summaries, reference_summaries),
                                     total=num_documents_to_summarize, desc="Scoring")):
    # Skip pairs where generation failed outright
    if not isinstance(gen, str) or gen.startswith("Error:") or not gen.strip():
        rows.append({
            "doc_index": i,
            "rouge1_f1": None, "rouge1_p": None, "rouge1_r": None,
            "rouge2_f1": None, "rouge2_p": None, "rouge2_r": None,
            "rougeL_f1": None, "rougeL_p": None, "rougeL_r": None,
            "bleu": None, "meteor": None,
            "bertscore_f1": None, "bertscore_p": None, "bertscore_r": None,
            "generated_summary": gen, "reference_summary": ref,
        })
        continue
 
    try:
        r1_f1, r1_p, r1_r = s.rouge_N(gen, ref, n=1)
        r2_f1, r2_p, r2_r = s.rouge_N(gen, ref, n=2)
        rl_f1, rl_p, rl_r = s.rouge_L(gen, ref)
        bleu_score = s.bleu(gen, ref, n=4)
        meteor_score = s.meteor(gen, ref)
        bs_f1, bs_p, bs_r = s.bert_score(gen, ref, model_type='roberta-large')
 
        rows.append({
            "doc_index": i,
            "rouge1_f1": r1_f1, "rouge1_p": r1_p, "rouge1_r": r1_r,
            "rouge2_f1": r2_f1, "rouge2_p": r2_p, "rouge2_r": r2_r,
            "rougeL_f1": rl_f1, "rougeL_p": rl_p, "rougeL_r": rl_r,
            "bleu": bleu_score, "meteor": meteor_score,
            "bertscore_f1": bs_f1, "bertscore_p": bs_p, "bertscore_r": bs_r,
            "generated_summary": gen, "reference_summary": ref,
        })
    except Exception as e:
        print(f"Error scoring document {i}: {e}")
        rows.append({
            "doc_index": i,
            "rouge1_f1": None, "rouge1_p": None, "rouge1_r": None,
            "rouge2_f1": None, "rouge2_p": None, "rouge2_r": None,
            "rougeL_f1": None, "rougeL_p": None, "rougeL_r": None,
            "bleu": None, "meteor": None,
            "bertscore_f1": None, "bertscore_p": None, "bertscore_r": None,
            "generated_summary": gen, "reference_summary": ref,
        })
 
results_df = pd.DataFrame(rows)
 
# -----------------------------------------------------------------------
# 5. Report averages + save
# -----------------------------------------------------------------------
metric_cols = ["rouge1_f1", "rouge1_p", "rouge1_r",
               "rouge2_f1", "rouge2_p", "rouge2_r",
               "rougeL_f1", "rougeL_p", "rougeL_r",
               "bleu", "meteor", "bertscore_f1", "bertscore_p", "bertscore_r"]
 
print("\n=== Average scores over", results_df[metric_cols].notna().all(axis=1).sum(),
      f"successfully scored documents (of {num_documents_to_summarize}) ===")
print(results_df[metric_cols].mean(numeric_only=True).round(4))
 
results_df.to_csv("qwen_summary_evaluation_results.csv", index=False)
print("\nFull per-document results saved to qwen_summary_evaluation_results.csv")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11907.20it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 24671.63it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UN


=== Average scores over 100 successfully scored documents (of 100) ===
rouge1_f1       0.2831
rouge1_p        0.3100
rouge1_r        0.2748
rouge2_f1       0.0778
rouge2_p        0.0853
rouge2_r        0.0763
rougeL_f1       0.1695
rougeL_p        0.1853
rougeL_r        0.1668
bleu            0.0688
meteor          0.2810
bertscore_f1    0.8603
bertscore_p     0.8686
bertscore_r     0.8524
dtype: float64

Full per-document results saved to qwen_summary_evaluation_results.csv
